In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import fashion_mnist
from matplotlib import pyplot as plt

tf.random.set_seed(1)
print("GPUs disponíveis: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

In [ ]:
# imagens reais
nomes = ["Camiseta", "Calça", "Pulôver", "Vestido", "Casaco",
         "Sandália", "Camisa", "Tênis", "Bolsa", "Bota"]

plt.figure(figsize=(8, 8))
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(x_train[i], cmap="Greys_r")
    plt.title(nomes[y_train[i]])
    plt.axis("off")
plt.show()

In [ ]:
# une treino e teste
dataset = np.concatenate([x_train, x_test], axis=0)

# normaliza
dataset = np.expand_dims(dataset, -1).astype("float32") / 255

In [ ]:
BATCH_SIZE = 64

dataset = tf.data.Dataset.from_tensor_slices(dataset)
dataset = dataset.shuffle(buffer_size=1024).batch(BATCH_SIZE)

In [ ]:
NOISE_DIM = 150

generator = keras.models.Sequential([
    keras.Input(shape=(NOISE_DIM,)),
    layers.Dense(7*7*256),
    layers.Reshape(target_shape=(7, 7, 256)),
    layers.Conv2DTranspose(256, 3, strides=2, padding="same"),
    layers.LeakyReLU(),
    layers.Conv2DTranspose(128, 3, strides=2, padding="same"),
    layers.LeakyReLU(),
    layers.Conv2DTranspose(1, 3, activation="sigmoid", padding="same")
], name="generator")

generator.summary()

In [ ]:
discriminator = keras.models.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(64, 3, strides=2, padding="same"),
    layers.LeakyReLU(),
    layers.Conv2D(128, 3, strides=2, padding="same"),
    layers.LeakyReLU(),
    layers.Flatten(),
    layers.Dense(1, activation="sigmoid")
], name="discriminator")

discriminator.summary()

In [ ]:
optimizerG = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
optimizerD = keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.5)

lossFn = keras.losses.BinaryCrossentropy()
gAccMetric = keras.metrics.BinaryAccuracy()
dAccMetric = keras.metrics.BinaryAccuracy()

In [ ]:
@tf.function
def trainDStep(data):
    batchSize = tf.shape(data)[0]
    noise = tf.random.normal(shape=(batchSize, NOISE_DIM))

    # 1 real, 0 falso
    y_true = tf.concat([tf.ones((batchSize, 1)), tf.zeros((batchSize, 1))], axis=0)

    with tf.GradientTape() as tape:
        fake = generator(noise)
        x = tf.concat([data, fake], axis=0)
        y_pred = discriminator(x)
        discriminatorLoss = lossFn(y_true, y_pred)

    grads = tape.gradient(discriminatorLoss, discriminator.trainable_weights)
    optimizerD.apply_gradients(zip(grads, discriminator.trainable_weights))
    dAccMetric.update_state(y_true, y_pred)

    return {"discriminator_loss": discriminatorLoss,
            "discriminator_accuracy": dAccMetric.result()}

In [ ]:
@tf.function
def trainGStep(data):
    batchSize = tf.shape(data)[0]
    noise = tf.random.normal(shape=(batchSize, NOISE_DIM))

    # quer ser classificado como real
    y_true = tf.ones((batchSize, 1))

    with tf.GradientTape() as tape:
        y_pred = discriminator(generator(noise))
        generatorLoss = lossFn(y_true, y_pred)

    grads = tape.gradient(generatorLoss, generator.trainable_weights)
    optimizerG.apply_gradients(zip(grads, generator.trainable_weights))
    gAccMetric.update_state(y_true, y_pred)

    return {"generator_loss": generatorLoss,
            "generator_accuracy": gAccMetric.result()}

In [ ]:
def plotarImagens(model, n=9):
    images = model(tf.random.normal(shape=(n * n, NOISE_DIM)))

    plt.figure(figsize=(9, 9))
    for i, image in enumerate(images):
        plt.subplot(n, n, i+1)
        plt.imshow(np.squeeze(image, -1), cmap="Greys_r")
        plt.axis("off")
    plt.show()

plotarImagens(generator)

In [ ]:
EPOCHS = 30
historico = {"g_loss": [], "d_loss": []}

for epoch in range(EPOCHS):
    dLossSum = gLossSum = dAccSum = gAccSum = 0.0
    cnt = 0

    for batch in dataset:
        d = trainDStep(batch)
        dLossSum += d["discriminator_loss"]
        dAccSum += d["discriminator_accuracy"]

        g = trainGStep(batch)
        gLossSum += g["generator_loss"]
        gAccSum += g["generator_accuracy"]

        cnt += 1

    historico["g_loss"].append(float(gLossSum / cnt))
    historico["d_loss"].append(float(dLossSum / cnt))

    print("Época: {}, Perda G: {:0.4f}, Perda D: {:0.4f}, Acc G: {:0.2f}%, Acc D: {:0.2f}%".format(
        epoch, gLossSum / cnt, dLossSum / cnt,
        100 * gAccSum / cnt, 100 * dAccSum / cnt))

    if (epoch + 1) % 10 == 0:
        plotarImagens(generator, n=5)

In [ ]:
plotarImagens(generator)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(historico["g_loss"], label="perda do gerador")
plt.plot(historico["d_loss"], label="perda do discriminador")
plt.xlabel("época")
plt.legend()
plt.show()